# Profile-Driven Document Table Pipeline

This notebook demonstrates a reusable core with document-specific configuration. Unknown or ambiguous layouts stop instead of being forced through the electricity profile.

## 1. Project setup and imports

In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import display

working_directory = Path.cwd()
project_root = (
    working_directory
    if (working_directory / "app").exists()
    else working_directory.parent
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import app.pipeline.builtin_profiles as builtin_profile_module
import app.pipeline.runner as pipeline_runner
from app.pipeline.profiles import TableTransformationProfile

builtin_profile_module = importlib.reload(builtin_profile_module)
pipeline_runner = importlib.reload(pipeline_runner)

BUILTIN_DOCUMENT_PROFILES = (
    builtin_profile_module.BUILTIN_DOCUMENT_PROFILES
)
detect_document_profile = pipeline_runner.detect_document_profile
run_profiled_pipeline = pipeline_runner.run_profiled_pipeline
transform_dataframe_with_profile = (
    pipeline_runner.transform_dataframe_with_profile
)

sample_documents = project_root / "sample_documents"
outputs = project_root / "outputs"
print("Project root:", project_root)

Project root: C:\Users\vinee\document-table-agent


## 2. Load a PDF without choosing page numbers

In [2]:
pdf_files = sorted(sample_documents.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError("No PDF files found in sample_documents.")

pdf_path = next(
    path
    for path in pdf_files
    if detect_document_profile(path, BUILTIN_DOCUMENT_PROFILES).name
    == "grid_india_weekly_report"
)
print("Using PDF:", pdf_path)

Using PDF: C:\Users\vinee\document-table-agent\sample_documents\Weekly 300326 to 050426_544 (1).pdf


## 3. Inspect registered profiles

In [3]:
profile_rows = []
for document_profile in BUILTIN_DOCUMENT_PROFILES:
    for table_profile in document_profile.tables:
        transformation = table_profile.transformation
        profile_rows.append(
            {
                "document profile": document_profile.name,
                "table profile": table_profile.name,
                "detection terms": document_profile.detection_terms,
                "search terms": table_profile.search_terms,
                "header rows": transformation.header_row_positions,
                "identity columns": transformation.identity_column_positions,
                "measure columns": transformation.measure_column_positions,
                "output schema": table_profile.output_schema,
                "postprocessors": tuple(
                    item.name for item in table_profile.postprocessors
                ),
            }
        )

display(pd.DataFrame(profile_rows))

,document profile,table profile,detection terms,search terms,header rows,identity columns,measure columns,output schema,postprocessors
0,grid_india_weekly_report,energy_consumption,"(Energy Consumption, Maximum Demand Met)","(Energy Consumption,)","(1,)","(0, 1)","(2, 3, 4, 5, 6, 7, 8)","OutputSchemaProfile(column_count=9, column_lev...","(long,)"
1,grid_india_weekly_report,maximum_demand,"(Energy Consumption, Maximum Demand Met)","(Maximum Demand Met,)","(1, 2)","(0, 1)","(2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15)","OutputSchemaProfile(column_count=16, column_le...","(long,)"
2,idsp_weekly_outbreak_report,current_outbreaks,"(WEEKLY OUTBREAK REPORT, Integrated Disease Su...","(Comments/ Action Taken,)","(0,)","(0, 1, 2, 3, 6, 7, 8, 9)","(4, 5)","OutputSchemaProfile(column_count=10, column_le...","(analysis,)"
3,idsp_weekly_outbreak_report,late_outbreaks,"(WEEKLY OUTBREAK REPORT, Integrated Disease Su...",(DISEASE OUTBREAKS OF PREVIOUS WEEKS REPORTED ...,"(0,)","(0, 1, 2, 3, 6, 7, 8)","(4, 5)","OutputSchemaProfile(column_count=9, column_lev...","(analysis,)"


## 4. Detect exactly one matching document profile

In [4]:
selected_profile = detect_document_profile(
    pdf_path, BUILTIN_DOCUMENT_PROFILES
)
print("Selected profile:", selected_profile.name)

Selected profile: grid_india_weekly_report


## 5. Run extraction, validation, transformation, mapping, and export

In [5]:
pipeline_result = run_profiled_pipeline(
    pdf_path,
    BUILTIN_DOCUMENT_PROFILES,
    output_dir=outputs,
    overwrite=True,
)
print("Completed profile:", pipeline_result.profile_name)

Completed profile: grid_india_weekly_report


## 6. Inspect pipeline results

In [6]:
summary_rows = []
for table_name, table_result in pipeline_result.tables.items():
    summary_rows.append(
        {
            "table profile": table_name,
            "located page": table_result.page_number,
            "raw shape": table_result.raw_table.shape,
            "transformed shape": table_result.transformed_table.shape,
            "column levels": table_result.transformed_table.columns.nlevels,
            "profile warnings": table_result.warnings,
            "output path": table_result.output_path,
            "derived shapes": {
                name: table.shape
                for name, table in table_result.postprocessed_tables.items()
            },
            "derived outputs": table_result.postprocessed_output_paths,
        }
    )
display(pd.DataFrame(summary_rows))

for table_name, table_result in pipeline_result.tables.items():
    print(table_name)
    display(table_result.transformed_table.head())

,table profile,located page,raw shape,transformed shape,column levels,profile warnings,output path,derived shapes,derived outputs
0,energy_consumption,4,"(42, 9)","(40, 9)",1,[],C:\Users\vinee\document-table-agent\outputs\cl...,"{'long': (280, 4)}",{'long': C:\Users\vinee\document-table-agent\o...
1,maximum_demand,3,"(42, 16)","(39, 16)",2,[],C:\Users\vinee\document-table-agent\outputs\cl...,"{'long': (273, 5)}",{'long': C:\Users\vinee\document-table-agent\o...


energy_consumption


,Region,States,30-03-2026,31-03-2026,01-04-2026,02-04-2026,03-04-2026,04-04-2026,05-04-2026
0,NR,Punjab,154.8,150.2,151.3,157.1,158.0,154.2,140.1
1,NR,Haryana,158.5,154.2,150.2,158.9,155.8,153.1,132.6
2,NR,Rajasthan,266.3,240.2,239.6,246.9,233.3,226.0,222.2
3,NR,Delhi,93.0,84.4,88.9,93.6,93.4,88.4,85.5
4,NR,UP,422.6,423.3,395.5,429.2,411.5,373.3,344.2


maximum_demand


Region       Date                     30-03-2026                   \
             States Max. Demand Met during the day Peak hr Shortage   
0     NR     Punjab                           7517                0   
1     NR    Haryana                           7940                0   
2     NR  Rajasthan                          12962                0   
3     NR      Delhi                           4456                0   
4     NR         UP                          21561                0   

                      31-03-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7441                0   
1                           7225                0   
2                          11326                0   
3                           4375                0   
4                          21301                0   

                      01-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7540                0   
1                           7944                0   
2                          11826                0   
3                           4367                0   
4                          22168                0   

                      02-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7579                0   
1                           8489                0   
2                          12382                0   
3                           4597                0   
4                          22797                0   

                      03-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7875                0   
1                           7529                0   
2                          11298                0   
3                           4445                0   
4                          21509                0   

                      04-04-2026                   \
  Max. Demand Met during the day Peak hr Shortage   
0                           7277                0   
1                           8302                0   
2                          11019                0   
3                           4244                0   
4                          19134                0   

                      05-04-2026                   
  Max. Demand Met during the day Peak hr Shortage  
0                           6607                0  
1                           7166                0  
2                          10966                0  
3                           3961                0  
4                          19252                0

## 7. Prove the transformer is not electricity-specific

The same pipeline transformation contract can process a synthetic sales layout with different headers and measures.

In [7]:
sales_raw = pd.DataFrame(
    [
        ["Quarterly sales", None, None],
        ["Product", "Units", "Revenue"],
        ["Books", "10", "125.50"],
        ["Games", "4", "80.00"],
    ]
)
sales_profile = TableTransformationProfile(
    header_row_positions=(1,),
    identity_column_positions=(0,),
    measure_column_positions=(1, 2),
)
sales_transformed = transform_dataframe_with_profile(
    sales_raw, sales_profile
)
display(sales_transformed)
display(sales_transformed.dtypes.to_frame("dtype"))

,Product,Units,Revenue
0,Books,10,125.5
1,Games,4,80.0


,dtype
Product,str
Units,int64
Revenue,float64


## 8. Adding another PDF family

1. Define a `TableTransformationProfile` for each table layout.
2. Wrap it in a `TableProfile` with search and table-selection rules.
3. Group tables in a `DocumentProfile` with distinctive detection terms.
4. Register the document profile and add synthetic plus representative-PDF tests.

No parser, validator, generic transformer, exporter, or pipeline-runner rewrite is required.

## 9. Findings

- The supplied electricity layout is one built-in profile, not logic embedded in the generic runner.
- Page numbers are located from profile search terms rather than hard-coded.
- Validation remains observational; explicit profile rules control transformation.
- Output schema checks catch unexpected column-count or header-level drift.
- Zero or multiple matching document profiles stop safely.
- New layouts are added through profiles and tests.